# 🔄 Základné pracovné postupy agentov s Microsoft Foundry (Python)

## 📋 Návod na orchestráciu pracovných procesov

Tento poznámkový blok predstavuje výkonné schopnosti **Workflow Builder** v Microsoft Agent Framework. Naučte sa vytvárať sofistikované, viacstupňové pracovné postupy agentov, ktoré dokážu zvládať komplexné obchodné procesy a plynulo koordinovať viaceré operácie AI.

> **Poznámka k migrácii:** Tento príklad predtým odkazoval na GitHub Models. GitHub Models sa prestávajú používať (ukončenie v júli 2026), preto teraz využíva **Microsoft Foundry** cez `FoundryChatClient`, ktorý cieli na Azure OpenAI **Responses API**.

## 🎯 Ciele učenia

### 🏗️ **Architektúra pracovných postupov**
- **Workflow Builder**: Navrhovanie a orchestrácia komplexných viacstupňových procesov
- **Spustenie na základe udalostí**: Spracovanie udalostí pracovného postupu a prechody stavmi
- **Vizualizácia pracovných postupov**: Vytváranie a vizualizácia štruktúr pracovných postupov
- **Integrácia Microsoft Foundry**: Využitie AI modelov v kontextoch pracovných postupov

### 🔄 **Orchestrácia procesov**
- **Sekvenčné operácie**: Reťazenie viacerých úloh agentov v logickom poradí
- **Podmienená logika**: Implementácia rozhodovacích bodov a vetvenia pracovných postupov
- **Spracovanie chýb**: Robustné zotavenie z chýb a odolnosť pracovných postupov
- **Správa stavov**: Sledovanie a riadenie stavu vykonávania pracovných postupov

### 📊 **Vzorce podnikových pracovných postupov**
- **Automatizácia obchodných procesov**: Automatizácia zložitých organizačných pracovných postupov
- **Koordinácia viacerých agentov**: Koordinácia viacerých špecializovaných agentov
- **Škálovateľné vykonávanie**: Navrhovanie pracovných postupov pre prevádzku na podnikovej úrovni
- **Monitorovanie a pozorovateľnosť**: Sledovanie výkonu a výsledkov pracovných postupov

## ⚙️ Predpoklady a nastavenie

### 📦 **Potrebné závislosti**

Nainštalujte Agent Framework so schopnosťami pracovných postupov:

```bash
pip install agent-framework -U
```

### 🔑 **Konfigurácia Microsoft Foundry**

Prihláste sa pomocou Azure CLI (`az login`), aby sa mohol použiť `AzureCliCredential` na autentifikáciu, potom nastavte detaily vášho projektu Microsoft Foundry.

**Nastavenie prostredia (.env súbor):**
```env
AZURE_AI_PROJECT_ENDPOINT=https://<your-project>.services.ai.azure.com
AZURE_AI_MODEL_DEPLOYMENT_NAME=gpt-5-mini
```

### 🏢 **Podnikové prípady použitia**

**Príklady obchodných procesov:**
- **Začlenenie zákazníka**: Viacstupňové overovacie a nastavovacie pracovné postupy
- **Príprava obsahu**: Automatizovaná tvorba, kontrola a publikovanie obsahu
- **Spracovanie údajov**: ETL pracovné postupy so AI-poháňanou transformáciou
- **Kontrola kvality**: Automatizované testovanie a validačné procesy

**Výhody pracovných postupov:**
- 🎯 **Spoľahlivosť**: Deterministické vykonávanie so zotavením z chýb
- 📈 **Škálovateľnosť**: Spracovanie automatizácie s veľkým objemom
- 🔍 **Pozorovateľnosť**: Kompletné auditné záznamy a monitorovanie
- 🔧 **Udržiavateľnosť**: Vizualizovaný dizajn a modulárne komponenty

## 🎨 Vzorce dizajnu pracovných postupov

### Základná štruktúra pracovného postupu
```mermaid
graph TD
    A[Začiatok] --> B[Úloha agenta 1]
    B --> C{Rozhodovací bod}
    C -->|Úspech| D[Úloha agenta 2]
    C -->|Neúspech| E[Spracovanie chýb]
    D --> F[Koniec]
    E --> F
```

**Kľúčové komponenty:**
- **WorkflowBuilder**: Hlavný orchestrátor
- **WorkflowEvent**: Spracovanie udalostí a komunikácia
- **WorkflowViz**: Vizualizácia pracovného postupu a ladenie

Postavme váš prvý inteligentný pracovný postup! 🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
# Core components for building sophisticated agent workflows
from agent_framework import WorkflowBuilder, WorkflowEvent, WorkflowViz
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential


In [ ]:
# 📦 Import Environment and System Utilities
# Essential libraries for configuration and environment management

import os                      # 🔧 Environment variable access
from dotenv import load_dotenv # 📁 Secure configuration loading

In [ ]:
# 🔧 Initialize Environment Configuration
# Load Microsoft Foundry project settings from .env file
load_dotenv()


In [ ]:
# Configure the Microsoft Foundry client with keyless authentication.
# FoundryChatClient targets the Azure OpenAI Responses API.
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


In [ ]:
REVIEWER_NAME = "Concierge"
REVIEWER_INSTRUCTIONS = """
    You are an are hotel concierge who has opinions about providing the most local and authentic experiences for travelers.
    The goal is to determine if the front desk travel agent has recommended the best non-touristy experience for a traveler.
    If so, state that it is approved.
    If not, provide insight on how to refine the recommendation without using a specific example. 
    """

In [ ]:
FRONTDESK_NAME = "FrontDesk"
FRONTDESK_INSTRUCTIONS = """
    You are a Front Desk Travel Agent with ten years of experience and are known for brevity as you deal with many customers.
    The goal is to provide the best activities and locations for a traveler to visit.
    Only provide a single recommendation per response.
    You're laser focused on the goal at hand.
    Don't waste time with chit chat.
    Consider suggestions when refining an idea.
    """

In [ ]:
reviewer_agent = provider.as_agent(
    name=REVIEWER_NAME,
    instructions=REVIEWER_INSTRUCTIONS,
)

front_desk_agent = provider.as_agent(
    name=FRONTDESK_NAME,
    instructions=FRONTDESK_INSTRUCTIONS,
)


In [ ]:
workflow = (
    WorkflowBuilder(start_executor=front_desk_agent)
    .add_edge(front_desk_agent, reviewer_agent)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra plus the graphviz system binary;
# fall back gracefully if it is not available.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
class DatabaseEvent(WorkflowEvent): ...

In [ ]:
# Display the exported workflow SVG inline in the notebook

from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")


In [ ]:
# Workflow.run_stream is no longer part of the public API; the current Workflow
# returns a results object whose `get_outputs()` produces the AgentResponse from
# each output executor. The reviewer (last stage) is the only output here.
events = await workflow.run("I would like to go to Paris.")
outputs = events.get_outputs()
result = outputs[0].text if outputs else ""

In [ ]:
result.replace("None", "")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vyhlásenie o zodpovednosti**:
Tento dokument bol preložený pomocou AI prekladateľskej služby [Co-op Translator](https://github.com/Azure/co-op-translator). Hoci sa snažíme o presnosť, vezmite prosím na vedomie, že automatické preklady môžu obsahovať chyby alebo nepresnosti. Pôvodný dokument v jeho natívnom jazyku by mal byť považovaný za autoritatívny zdroj. Pre kritické informácie sa odporúča profesionálny ľudský preklad. Nie sme zodpovední za žiadne nedorozumenia alebo nesprávne interpretácie vyplývajúce z použitia tohto prekladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
